### Dowload dataset from ZENODO platform

dowload the dataset from Zenodo and extract it from the zip

In [1]:
import hashlib, io, zipfile, requests
from pathlib import Path

ZENODO_URL = "https://zenodo.org/records/1323611/files/data.zip?download=1"
ZENODO_MD5 = "ac0b28f34686660219c4e43417053f9e"
DATA_DIR = Path("data_hydraulic")

def download_dataset(url=ZENODO_URL, md5=ZENODO_MD5, data_dir=DATA_DIR):
    if data_dir.exists() and any(data_dir.rglob("PS1.txt")):
        return data_dir                                   # già scaricato
    data_dir.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    if hashlib.md5(r.content).hexdigest() != md5:
        raise ValueError("Checksum MD5 not matching: download corrupted?")
    zipfile.ZipFile(io.BytesIO(r.content)).extractall(data_dir)
    return data_dir

### Load dataset and first check

In [2]:
# Download and extract the dataset if it is not already available
dataset_dir = download_dataset()
print(f"Dataset available in: {dataset_dir}")

Dataset available in: data_hydraulic


In [3]:
import pandas as pd

# Sensor files: each row is one 60s cycle, each column a time sample within that cycle.
SENSOR_FILES = [
    "PS1", "PS2", "PS3", "PS4", "PS5", "PS6",  # pressure, 100 Hz
    "EPS1",                                     # motor power, 100 Hz
    "FS1", "FS2",                               # volume flow, 10 Hz
    "TS1", "TS2", "TS3", "TS4",                 # temperature, 1 Hz
    "VS1",                                      # vibration, 1 Hz
    "CE", "CP", "SE",                           # cooling efficiency/power/factor, 1 Hz
]
PROFILE_COLUMNS = ["cooler_pct", "valve_pct", "pump_leak", "accumulator_bar", "stable_flag"]


def load_sensor(name: str, data_dir=dataset_dir) -> pd.DataFrame:
    return pd.read_csv(data_dir / f"{name}.txt", sep="\t", header=None)


sensors = {name: load_sensor(name) for name in SENSOR_FILES}
profile = pd.read_csv(dataset_dir / "profile.txt", sep="\t", header=None, names=PROFILE_COLUMNS)

print(f"Loaded {len(sensors)} sensor files and profile.txt ({profile.shape[0]} cycles)")

Loaded 17 sensor files and profile.txt (2205 cycles)


### Shape and dtype overview

In [4]:
for name, df in sensors.items():
    print(f"{name:>5}: shape={df.shape}, dtype={df.dtypes.unique()}")

print(f"\nprofile: shape={profile.shape}, dtypes=\n{profile.dtypes}")

  PS1: shape=(2205, 6000), dtype=[dtype('float64')]
  PS2: shape=(2205, 6000), dtype=[dtype('float64')]
  PS3: shape=(2205, 6000), dtype=[dtype('float64') dtype('int64')]
  PS4: shape=(2205, 6000), dtype=[dtype('float64')]
  PS5: shape=(2205, 6000), dtype=[dtype('float64')]
  PS6: shape=(2205, 6000), dtype=[dtype('float64')]
 EPS1: shape=(2205, 6000), dtype=[dtype('float64')]
  FS1: shape=(2205, 600), dtype=[dtype('float64')]
  FS2: shape=(2205, 600), dtype=[dtype('float64')]
  TS1: shape=(2205, 60), dtype=[dtype('float64')]
  TS2: shape=(2205, 60), dtype=[dtype('float64')]
  TS3: shape=(2205, 60), dtype=[dtype('float64')]
  TS4: shape=(2205, 60), dtype=[dtype('float64')]
  VS1: shape=(2205, 60), dtype=[dtype('float64')]
   CE: shape=(2205, 60), dtype=[dtype('float64')]
   CP: shape=(2205, 60), dtype=[dtype('float64')]
   SE: shape=(2205, 60), dtype=[dtype('float64') dtype('int64')]

profile: shape=(2205, 5), dtypes=
cooler_pct         int64
valve_pct          int64
pump_leak          

### Condition labels overview (`profile.txt`)

Each row of `profile.txt` gives the ground-truth operating condition for the
corresponding cycle: cooler condition (%), valve condition (%), internal pump
leakage (0/1/2), hydraulic accumulator pressure (bar), and whether the cycle
was stable (0/1).

In [5]:
display(profile.describe())

for col in PROFILE_COLUMNS:
    print(f"\n{col} value counts:")
    print(profile[col].value_counts().sort_index())

,cooler_pct,valve_pct,pump_leak,accumulator_bar,stable_flag
count,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000
mean,41.240816,90.693878,0.669388,107.199546,0.342857
std,42.383143,10.681802,0.817233,16.435848,0.474772
min,3.000000,73.000000,0.000000,90.000000,0.000000
25%,3.000000,80.000000,0.000000,90.000000,0.000000
50%,20.000000,100.000000,0.000000,100.000000,0.000000
75%,100.000000,100.000000,1.000000,130.000000,1.000000
max,100.000000,100.000000,2.000000,130.000000,1.000000



cooler_pct value counts:
cooler_pct
3      732
20     732
100    741
Name: count, dtype: int64

valve_pct value counts:
valve_pct
73      360
80      360
90      360
100    1125
Name: count, dtype: int64

pump_leak value counts:
pump_leak
0    1221
1     492
2     492
Name: count, dtype: int64

accumulator_bar value counts:
accumulator_bar
90     808
100    399
115    399
130    599
Name: count, dtype: int64

stable_flag value counts:
stable_flag
0    1449
1     756
Name: count, dtype: int64


### Sensor signal overview

A `.describe()` over the raw sample columns of a high-rate sensor (e.g. 6000
columns for `PS1`) is not very readable, so instead we describe the per-cycle
mean and standard deviation — one summary value per cycle.

In [6]:
for name in ["PS1", "TS1"]:
    df = sensors[name]
    print(f"=== {name}: per-cycle mean ===")
    display(df.mean(axis=1).describe())
    print(f"=== {name}: per-cycle std ===")
    display(df.std(axis=1).describe())

=== PS1: per-cycle mean ===


count    2205.000000
mean      160.485315
std         4.699425
min       155.391547
25%       158.100195
50%       158.960895
75%       161.000735
max       180.922708
dtype: float64

=== PS1: per-cycle std ===


count    2205.000000
mean       15.296219
std         2.068116
min        13.927662
25%        14.144613
50%        14.730968
75%        15.050131
max        22.141793
dtype: float64

=== TS1: per-cycle mean ===


count    2205.000000
mean       45.424567
std         7.991933
min        35.313783
25%        36.237150
50%        44.836650
75%        54.104317
max        57.899283
dtype: float64

=== TS1: per-cycle std ===


count    2205.000000
mean        0.192950
std         0.044715
min         0.060857
25%         0.155940
50%         0.194232
75%         0.214453
max         0.585740
dtype: float64

# Creation of the dataset needed for the documentation
File containing 
- 10 cycle of 2 sensors: PS1 and EPS1
- only stable cycles: `stables == 0`
- cycles downsampled to 200 points x cycle

In [ ]:
import numpy as np

DOC_SENSORS = ["PS1", "EPS1"]
N_CYCLES = 10
N_SAMPLES_OUT = 200


# Healthy baseline: all components in optimal condition, and a stable (steady-state) cycle
healthy_stable = (
    (profile["cooler_pct"] == 100)
    & (profile["valve_pct"] == 100)
    & (profile["pump_leak"] == 0)
    & (profile["accumulator_bar"] == 130)
    & (profile["stable_flag"] == 0)
).to_numpy()

# Find the first run of N_CYCLES consecutive healthy + stable cycles
run_start = None
for i in range(len(healthy_stable) - N_CYCLES + 1):
    if healthy_stable[i : i + N_CYCLES].all():
        run_start = i
        break
if run_start is None:
    raise ValueError(f"No run of {N_CYCLES} consecutive healthy stable cycles found")

cycle_idx = list(range(run_start, run_start + N_CYCLES))
print(f"Using cycles {cycle_idx[0]}-{cycle_idx[-1]} (healthy components, stable_flag == 0)")

# Downsample each cycle (6000 samples) to N_SAMPLES_OUT via block averaging
rows = []
for sensor_name in DOC_SENSORS:
    cycles = sensors[sensor_name].iloc[cycle_idx].to_numpy()
    n_samples_in = cycles.shape[1]
    assert n_samples_in % N_SAMPLES_OUT == 0, f"{sensor_name}: {n_samples_in} not divisible by {N_SAMPLES_OUT}"
    downsampled = cycles.reshape(len(cycle_idx), N_SAMPLES_OUT, -1).mean(axis=2)
    for cycle, values in zip(cycle_idx, downsampled):
        rows.append({"sensor": sensor_name, "cycle": cycle, **{str(j): v for j, v in enumerate(values)}})

doc_df = pd.DataFrame(rows)
doc_df.to_csv("final_documentation/dataset.txt", sep="\t", index=False)

print(f"Saved final_documentation/dataset.txt with shape {doc_df.shape}")
display(doc_df.iloc[:, :5])